# Analisis de `datos.json` para la media móvil optimizada

In [ ]:
# Figuras VECTORIALES en vez de PNG. Tres motivos, en orden de importancia:
#   · git guarda cada blob comprimido, y un SVG de líneas comprime 5 veces
#     mejor que el mismo gráfico en PNG (150 KB contra 761 KB este notebook);
#   · se puede ampliar sin que se pixele;
#   · apagando la fecha que matplotlib incrusta en el SVG, dos ejecuciones
#     seguidas con los mismos datos dan bytes IDÉNTICOS — así reejecutar el
#     notebook no crea una copia nueva en la historia del repositorio.
%config InlineBackend.figure_formats = ['svg']
%config InlineBackend.print_figure_kwargs = {'metadata': {'Date': None}}

import json
import matplotlib.pyplot as plt
from matplotlib.ticker import LogLocator

plt.rcParams["svg.hashsalt"] = "modulo05"   # ids estables entre corridas
plt.rcParams["figure.figsize"] = (10, 5.5)
plt.rcParams["axes.grid"] = True

# ── Ayudantes comunes a todas las secciones ─────────────────────────────────
#

print("Ayudantes cargados")

## Gráfico de las señales

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

# 1. Leer el archivo JSON directamente a un DataFrame de Pandas
df = pd.read_json('./build/datos.json')
df.set_index('ix', inplace=True)

# 2. Graficar todo junto de forma dinámica
plt.figure(figsize=(64, 7))

# Graficamos la señal original más tenue
plt.plot(df['y'], label='Original', alpha=0.3, color='gray', linewidth=1.0)

# Buscamos todas las columnas que correspondan a medias móviles y las graficamos en bucle
columnas_mm = [col for col in df.columns if col.startswith('media_movil_')]

# Opcional: Ordenarlas numéricamente por el período para que la leyenda quede ordenada
columnas_mm.sort(key=lambda x: int(x.split('_p')[1]))

for col in columnas_mm:
    # Extraemos el número del período para una etiqueta limpia (ej: "P2", "P64")
    periodo_label = col.split('_')[2].upper() 
    plt.plot(df[col], label=f'Media Móvil {periodo_label}', linewidth=1.4)

plt.title('Comparativa de Medias Móviles Concurrentes (CUDA)')
plt.xlabel('Muestras')
plt.ylabel('Amplitud')
plt.legend(loc='upper right')
plt.grid(True)
plt.show()


## Histograma de las señales

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np

# 1. Leer el archivo JSON directamente a un DataFrame de Pandas
df = pd.read_json('./build/datos.json')
df.set_index('ix', inplace=True)

# 2. Detectar columnas de medias móviles y ordenarlas numéricamente
columnas_mm = [col for col in df.columns if col.startswith('media_movil_')]
columnas_mm.sort(key=lambda x: int(x.split('_p')[1]))

# 3. Detectar dinámicamente el periodo máximo para el recorte del Warm-up
periodo_maximo = int(columnas_mm[-1].split('_p')[1])

# Filtrar los datos a partir del índice del periodo máximo para el análisis estable
df_estables = df.loc[periodo_maximo:]

# 4. Graficar histogramas dinámicos
plt.figure(figsize=(12, 7))

# Calculamos dinámicamente los límites basados en la señal original en el tramo estable
min_val = df_estables['y'].min()
max_val = df_estables['y'].max()
clases = np.linspace(min_val, max_val, 129) # 128 clases

# Graficamos el histograma de la señal original
plt.hist(df_estables['y'], bins=clases, alpha=0.3, color='gray', edgecolor='black', label=f'Original (>={periodo_maximo})')

# Graficamos automáticamente cada media móvil que venga en el JSON
for col in columnas_mm:
    periodo_label = col.split('_')[2].upper()
    plt.hist(df_estables[col], bins=clases, alpha=0.2, histtype='bar', linewidth=2, label=f'{periodo_label} (>={periodo_maximo})')

plt.title('Distribución de la Señal (128 Clases)')
plt.xlabel('Amplitud')
plt.xticks(np.arange(0, 1.1, 0.1))
plt.ylabel('Frecuencia (Conteos)')
plt.legend(loc='upper right')
plt.grid(True)

plt.tight_layout()
plt.show()


## Análisis del Decaimiento de la Varianza vs. Período

Según la teoría estadística, al promediar $N$ variables aleatorias independientes e idénticamente distribuidas (i.i.d.), la varianza del promedio debería reducirse de forma inversamente proporcional al tamaño de la muestra ($V_{mm} = \frac{V_{original}}{N}$). 

En esta sección se realiza lo siguiente de forma 100% dinámica:
1. Se extraen numéricamente los períodos ($N$) de las columnas disponibles en el archivo JSON.
2. Se calcula la varianza real de cada media móvil en su tramo estable (evitando el sesgo de inicialización).
3. Se compara la varianza experimental obtenida por la GPU contra la curva teórica esperada.


In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np

# 1. Leer el archivo JSON directamente a un DataFrame de Pandas
df = pd.read_json('./build/datos.json')
df.set_index('ix', inplace=True)

# 2. Detectar columnas de medias móviles y extraer sus períodos numéricos
columnas_mm = [col for col in df.columns if col.startswith('media_movil_')]

# Creamos una lista de tuplas (periodo_entero, nombre_columna) y la ordenamos
lista_periodos = [(int(col.split('_p')[-1]), col) for col in columnas_mm]
lista_periodos.sort(key=lambda x: x[0])

# Separamos en dos listas ordenadas para trabajar
periodos = [p[0] for p in lista_periodos]
columnas_ordenadas = [p[1] for p in lista_periodos]

# 3. Calcular el Warm-up máximo para limpiar el análisis estadístico
periodo_maximo = periodos[-1]
df_estables = df.loc[periodo_maximo:]

# 4. Calcular varianzas (La teórica de una distribución uniforme entre 0 y 1 es 1/12)
varianza_original = df_estables['y'].var()
varianzas_experimentales = [df_estables[col].var() for col in columnas_ordenadas]

# Curva teórica de decaimiento: Varianza_Original / N
periodos_teoricos = np.linspace(periodos[0], periodos[-1], 500)
varianzas_teoricas = varianza_original / periodos_teoricos

# 5. Graficar la comparativa
plt.figure(figsize=(10, 6))

# Graficamos la curva teórica continua
plt.plot(periodos_teoricos, varianzas_teoricas, 
         label='Decaimiento Teórico ($V_{orig} / N$)', 
         color='red', linestyle='--', linewidth=2)

# Graficamos los puntos reales obtenidos de la GPU
plt.scatter(periodos, varianzas_experimentales, 
            color='dodgerblue', s=80, zorder=5, 
            label='Varianza Experimental (GPU)')

# Anotaciones estéticas
plt.title('Decaimiento de la Varianza en función del Período de la Media Móvil')
plt.xlabel('Período de la ventana ($N$)')
plt.ylabel('Varianza de la señal promediada')
plt.xticks(periodos) # Ajusta el eje X exactamente a tus hiperparámetros
plt.legend(loc='upper right')
plt.grid(True, which="both", ls="-")

plt.tight_layout()
plt.show()